# ARC2 Hybrid v2 — Fast Symbolic Pre-Pass + Loss-Aware LoRA TTT + Structural Invariants

This is the **V2 architecture** for the ARC Prize 2026 (ARC-AGI-2), building on the 32.22% NVARC baseline.

### Key V2 Innovations:
1. **Fast Symbolic Pre-Pass Engine (`arc_symbolic.py`)**:
   - Runs deterministic rules (D4 symmetries, 1-to-1 color remappings, object/connected component cropping, tiling, scaling, gravity) on CPU in sub-second time.
   - Tasks with 100% verified exact rules across all training demonstrations are immediately solved, bypassing heavy GPU compute and reserving massive budget for hard puzzles.
2. **Structural Invariant & Geometry Verification (`arc_invariants.py`)**:
   - Infers mathematical shape relations ($H_{\text{out}} \times W_{\text{out}}$) and color palette bounds.
   - Prunes/penalizes candidate grids with illegal dimensions or hallucinated colors.
3. **Loss-Aware Adaptive Test-Time Training (`arc_solver.py`)**:
   - Stops LoRA fine-tuning early if demonstration loss drops below $5\times 10^{-4}$, saving 30–50% of TTT time on straightforward tasks.
4. **Diverse Hypothesis Attempt Selection (`arc_decoder.py`)**:
   - `attempt_1`: Best overall consensus prediction (exact symbolic or top neural).
   - `attempt_2`: Best distinct alternative candidate with non-zero structural distance to maximize the probability of solving via either attempt.
5. **Multi-GPU Adaptive Budget Scheduler (`starter.py`)**:
   - Cheap-first ordering + 2-phase deep search on starved/uncertain tasks within Kaggle's 12-hour limit.


In [ ]:
# ---------------------------------------------------------------------------
# Global wall-clock budget. Rerun (hidden test): 12h minus a 10 min write buffer.
# ---------------------------------------------------------------------------
import os, time, json, glob
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
T0 = time.time()

def has_test_challenges():
    candidates = [
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json",
        "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json",
        "/kaggle/input/arc-prize-2026/arc-agi_test_challenges.json",
        "/kaggle/input/arc-agi_test_challenges.json",
    ]
    return any(os.path.exists(c) for c in candidates) or bool(glob.glob("/kaggle/input/**/arc*test_challenges.json", recursive=True))

if RERUN:
    global_end_time = T0 + 12 * 3600 - 600
elif has_test_challenges():
    # If test file is available in interactive mode, allocate 11.5 hours
    global_end_time = T0 + 11.5 * 3600 - 600
else:
    global_end_time = T0 + 55 * 60  # 55-minute interactive debug run
print(f"rerun={RERUN} budget={(global_end_time-T0)/3600:.2f}h cutoff={time.ctime(global_end_time)}")


In [ ]:
!pip uninstall -y tensorflow torchao
import sys, os, glob, site
# 1. Put site-packages at the front so real PyTorch is loaded
for sp in reversed(site.getsitepackages()):
    if sp in sys.path:
        sys.path.remove(sp)
    sys.path.insert(0, sp)

# 2. Move utility scripts to the end of sys.path so unsloth is available
for p in list(sys.path):
    if 'pip_install_unsloth_flash_patch' in p or 'usr/lib/notebooks' in p:
        sys.path.remove(p)
        sys.path.append(p)

if 'torchao' in sys.modules:
    del sys.modules['torchao']

import torch
print(sys.version.split()[0], "torch", torch.__version__, "gpus", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("models:", glob.glob("/kaggle/input/models/*/*"), glob.glob("/kaggle/input/qwen*"))
print("utility scripts:", [p for p in sys.path if "/kaggle/usr/lib" in p][:5])
os.makedirs("/kaggle/working/logs", exist_ok=True)
os.makedirs("/kaggle/working/symbolic_outputs", exist_ok=True)
os.makedirs("/kaggle/inference_outputs", exist_ok=True)
os.makedirs("/kaggle/inference_outputs_deep", exist_ok=True)


In [ ]:
%%writefile arc_invariants.py
"""
arc_invariants.py - ARC-specific structural & invariant verification engine
"""
import numpy as np
from collections import Counter
from typing import List, Tuple, Optional, Set, Dict, Any

def shape(grid) -> Tuple[int, int]:
    if isinstance(grid, np.ndarray):
        return grid.shape[0], grid.shape[1]
    if not isinstance(grid, (list, tuple)) or len(grid) == 0:
        return 0, 0
    return len(grid), len(grid[0]) if isinstance(grid[0], (list, tuple, np.ndarray)) else 0

def is_valid_grid(grid) -> bool:
    """Strictly validates if a grid is a valid ARC 2D discrete grid (max 30x30, ints 0-9)."""
    if grid is None:
        return False
    if isinstance(grid, np.ndarray):
        if grid.ndim != 2:
            return False
        h, w = grid.shape
        if h == 0 or w == 0 or h > 30 or w > 30:
            return False
        if not np.issubdtype(grid.dtype, np.integer):
            return False
        if grid.min() < 0 or grid.max() > 9:
            return False
        return True
    elif isinstance(grid, list):
        if len(grid) == 0 or len(grid) > 30:
            return False
        if not all(isinstance(row, list) and len(row) > 0 for row in grid):
            return False
        w = len(grid[0])
        if w == 0 or w > 30:
            return False
        for row in grid:
            if len(row) != w:
                return False
            for val in row:
                if not isinstance(val, (int, np.integer)) or not (0 <= val <= 9):
                    return False
        return True
    return False

def to_np_grid(grid) -> Optional[np.ndarray]:
    if grid is None:
        return None
    if isinstance(grid, np.ndarray):
        if is_valid_grid(grid):
            return grid.astype(int)
        return None
    if is_valid_grid(grid):
        return np.array(grid, dtype=int)
    return None

def infer_shape_rule(train_pairs: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Infers the mathematical relationship between input grid shapes and output grid shapes.
    Possible rules:
    - 'same': Output shape is identical to input shape (H_out == H_in, W_out == W_in)
    - 'constant': Output shape is constant across all examples (e.g. 3x3)
    - 'scale': Output shape is a fixed integer multiple of input shape (e.g. 2x, 3x)
    - 'fraction': Output shape is a fixed fraction (e.g. 1/2, 1/3)
    - 'transpose': Output shape is transposed (H_out == W_in, W_out == H_in)
    - 'unknown': Variable/dynamic shape
    """
    if not train_pairs:
        return {"rule": "unknown"}

    in_shapes = [shape(p["input"]) for p in train_pairs]
    out_shapes = [shape(p["output"]) for p in train_pairs]

    # Check 'same'
    if all(si == so for si, so in zip(in_shapes, out_shapes)):
        return {"rule": "same"}

    # Check 'constant'
    if len(set(out_shapes)) == 1:
        return {"rule": "constant", "shape": out_shapes[0]}

    # Check 'scale'
    h_ratios = [so[0] / si[0] if si[0] > 0 else 0 for si, so in zip(in_shapes, out_shapes)]
    w_ratios = [so[1] / si[1] if si[1] > 0 else 0 for si, so in zip(in_shapes, out_shapes)]
    if len(set(h_ratios)) == 1 and len(set(w_ratios)) == 1:
        hr, wr = h_ratios[0], w_ratios[0]
        if hr > 0 and wr > 0:
            return {"rule": "scale", "scale_h": hr, "scale_w": wr}

    # Check 'transpose'
    if all(si[0] == so[1] and si[1] == so[0] for si, so in zip(in_shapes, out_shapes)):
        return {"rule": "transpose"}

    return {"rule": "unknown"}

def predict_output_shape(shape_rule: Dict[str, Any], test_input) -> Optional[Tuple[int, int]]:
    """Predicts expected output shape given the inferred shape rule and test input."""
    hin, win = shape(test_input)
    rule = shape_rule.get("rule", "unknown")
    if rule == "same":
        return hin, win
    elif rule == "constant":
        return shape_rule["shape"]
    elif rule == "scale":
        sh = int(round(hin * shape_rule["scale_h"]))
        sw = int(round(win * shape_rule["scale_w"]))
        if 0 < sh <= 30 and 0 < sw <= 30:
            return sh, sw
    elif rule == "transpose":
        return win, hin
    return None

def infer_allowed_palette(train_pairs: List[Dict[str, Any]], test_input) -> Set[int]:
    """
    Determines the set of permitted colors for the test output.
    In almost all ARC tasks, the output colors must be:
    - Colors appearing in the test input, OR
    - Fixed constant colors that appeared across all training outputs.
    """
    test_in_colors = set(np.unique(np.asarray(test_input))) if isinstance(test_input, (list, np.ndarray)) else set()
    train_out_colors = set()
    for p in train_pairs:
        train_out_colors.update(np.unique(np.asarray(p["output"])))

    # The allowed palette is the union of test input colors and training output colors
    return test_in_colors | train_out_colors

def validate_candidate_invariants(
    candidate: np.ndarray,
    train_pairs: List[Dict[str, Any]],
    test_input: np.ndarray,
    shape_rule: Optional[Dict[str, Any]] = None
) -> Tuple[bool, float]:
    """
    Evaluates how well a candidate adheres to ARC structural invariants.
    Returns:
      (is_plausible: bool, penalty_score: float)
    """
    if not is_valid_grid(candidate):
        return False, 1000.0

    penalty = 0.0
    cand_h, cand_w = candidate.shape

    if shape_rule is None:
        shape_rule = infer_shape_rule(train_pairs)

    expected_shape = predict_output_shape(shape_rule, test_input)
    if expected_shape is not None:
        exp_h, exp_w = expected_shape
        if (cand_h, cand_w) != (exp_h, exp_w):
            # Heavy penalty for shape mismatch when shape rule is clear
            penalty += 50.0 + 5.0 * (abs(cand_h - exp_h) + abs(cand_w - exp_w))

    # Palette validation
    allowed_palette = infer_allowed_palette(train_pairs, test_input)
    cand_colors = set(np.unique(candidate))
    disallowed_colors = cand_colors - allowed_palette
    if disallowed_colors:
        # Severe penalty for hallucinating colors not present in problem
        penalty += 30.0 * len(disallowed_colors)

    return penalty < 40.0, penalty

def grid_hamming_distance(grid1: np.ndarray, grid2: np.ndarray) -> float:
    """Computes normalized distance between two grids for attempt diversification."""
    if grid1.shape != grid2.shape:
        return 1.0
    if grid1.size == 0:
        return 0.0
    return float(np.mean(grid1 != grid2))


In [ ]:
%%writefile arc_symbolic.py
"""
arc_symbolic.py - Fast Pure-Python/NumPy Deterministic & Symbolic ARC Rule Engine
"""
import numpy as np
from collections import Counter, deque
from typing import List, Dict, Any, Tuple, Optional, Callable

from arc_invariants import is_valid_grid, to_np_grid

def get_shape(grid: np.ndarray) -> Tuple[int, int]:
    return grid.shape[0], grid.shape[1]

def get_most_common_color(grid: np.ndarray) -> int:
    counts = np.bincount(grid.ravel(), minlength=10)
    return int(np.argmax(counts))

def get_least_common_color(grid: np.ndarray, exclude_bg: bool = True) -> int:
    counts = np.bincount(grid.ravel(), minlength=10)
    if exclude_bg:
        bg = get_most_common_color(grid)
        counts[bg] = 0
    non_zero = np.where(counts > 0)[0]
    if len(non_zero) == 0:
        return 0
    return int(non_zero[np.argmin(counts[non_zero])])

def get_non_bg_cells(grid: np.ndarray, bg: Optional[int] = None) -> List[Tuple[int, int]]:
    if bg is None:
        bg = get_most_common_color(grid)
    coords = np.argwhere(grid != bg)
    return [tuple(c) for c in coords]

def bbox_cells(cells: List[Tuple[int, int]]) -> Optional[Tuple[int, int, int, int]]:
    if not cells:
        return None
    rs = [r for r, c in cells]
    cs = [c for r, c in cells]
    return min(rs), max(rs), min(cs), max(cs)

def crop_bbox(grid: np.ndarray, bbox: Tuple[int, int, int, int]) -> np.ndarray:
    r1, r2, c1, c2 = bbox
    return grid[r1:r2+1, c1:c2+1].copy()

def crop_non_bg(grid: np.ndarray, bg: Optional[int] = None) -> np.ndarray:
    if bg is None:
        bg = get_most_common_color(grid)
    cells = get_non_bg_cells(grid, bg)
    if not cells:
        return np.array([[bg]], dtype=int)
    bbox = bbox_cells(cells)
    return crop_bbox(grid, bbox)

# ----------------- Connected Components -----------------
def get_connected_components(grid: np.ndarray, bg: Optional[int] = None, diag: bool = False) -> List[Dict[str, Any]]:
    if bg is None:
        bg = get_most_common_color(grid)
    h, w = grid.shape
    seen = np.zeros((h, w), dtype=bool)
    comps = []
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if diag:
        neighbors += [(-1, -1), (-1, 1), (1, -1), (1, 1)]

    for r in range(h):
        for c in range(w):
            if seen[r, c] or grid[r, c] == bg:
                continue
            color = int(grid[r, c])
            q = deque([(r, c)])
            seen[r, c] = True
            cells = []
            while q:
                x, y = q.popleft()
                cells.append((x, y))
                for dx, dy in neighbors:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < h and 0 <= ny < w and not seen[nx, ny] and grid[nx, ny] == color:
                        seen[nx, ny] = True
                        q.append((nx, ny))
            comps.append({"color": color, "cells": cells})
    return comps

def crop_largest_component(grid: np.ndarray, bg: Optional[int] = None) -> np.ndarray:
    comps = get_connected_components(grid, bg)
    if not comps:
        return crop_non_bg(grid, bg)
    largest = max(comps, key=lambda z: len(z["cells"]))
    bbox = bbox_cells(largest["cells"])
    return crop_bbox(grid, bbox)

def crop_smallest_component(grid: np.ndarray, bg: Optional[int] = None) -> np.ndarray:
    comps = get_connected_components(grid, bg)
    if not comps:
        return crop_non_bg(grid, bg)
    smallest = min(comps, key=lambda z: len(z["cells"]))
    bbox = bbox_cells(smallest["cells"])
    return crop_bbox(grid, bbox)

# ----------------- Symmetries & Geometric Primitives -----------------
def sym_identity(g: np.ndarray) -> np.ndarray:
    return g.copy()

def sym_rot90(g: np.ndarray) -> np.ndarray:
    return np.rot90(g, k=-1).copy()

def sym_rot180(g: np.ndarray) -> np.ndarray:
    return np.rot90(g, k=2).copy()

def sym_rot270(g: np.ndarray) -> np.ndarray:
    return np.rot90(g, k=1).copy()

def sym_flip_h(g: np.ndarray) -> np.ndarray:
    return np.fliplr(g).copy()

def sym_flip_v(g: np.ndarray) -> np.ndarray:
    return np.flipud(g).copy()

def sym_transpose(g: np.ndarray) -> np.ndarray:
    return np.transpose(g).copy()

def sym_antitranspose(g: np.ndarray) -> np.ndarray:
    return np.rot90(np.flipud(g), k=1).copy()

GEOMETRIC_TRANSFORMS = [
    ("identity", sym_identity),
    ("rot90", sym_rot90),
    ("rot180", sym_rot180),
    ("rot270", sym_rot270),
    ("flip_h", sym_flip_h),
    ("flip_v", sym_flip_v),
    ("transpose", sym_transpose),
    ("antitranspose", sym_antitranspose),
]

# ----------------- Border and Subgrid Extraction -----------------
def crop_remove_border(grid: np.ndarray) -> Optional[np.ndarray]:
    h, w = grid.shape
    if h > 2 and w > 2:
        return grid[1:h-1, 1:w-1].copy()
    return None

def extract_outer_border(grid: np.ndarray) -> np.ndarray:
    bg = get_most_common_color(grid)
    h, w = grid.shape
    out = np.full_like(grid, bg)
    out[0, :] = grid[0, :]
    out[h-1, :] = grid[h-1, :]
    out[:, 0] = grid[:, 0]
    out[:, w-1] = grid[:, w-1]
    return out

# ----------------- Symmetry Completion -----------------
def complete_symmetry_h(grid: np.ndarray) -> np.ndarray:
    """Completes horizontal symmetry by overlaying left half or right half."""
    h, w = grid.shape
    mid = w // 2
    out = grid.copy()
    # Mirror left to right
    for r in range(h):
        for c in range(mid):
            if out[r, c] != 0 and out[r, w - 1 - c] == 0:
                out[r, w - 1 - c] = out[r, c]
            elif out[r, w - 1 - c] != 0 and out[r, c] == 0:
                out[r, c] = out[r, w - 1 - c]
    return out

def complete_symmetry_v(grid: np.ndarray) -> np.ndarray:
    """Completes vertical symmetry by overlaying top half or bottom half."""
    h, w = grid.shape
    mid = h // 2
    out = grid.copy()
    for r in range(mid):
        for c in range(w):
            if out[r, c] != 0 and out[h - 1 - r, c] == 0:
                out[h - 1 - r, c] = out[r, c]
            elif out[h - 1 - r, c] != 0 and out[r, c] == 0:
                out[r, c] = out[h - 1 - r, c]
    return out

# ----------------- Color Remapping -----------------
def replace_colors(grid: np.ndarray, mapping: Dict[int, int]) -> np.ndarray:
    out = grid.copy()
    for src, dst in mapping.items():
        out[grid == src] = dst
    return out

def infer_exact_color_map(train_pairs: List[Dict[str, Any]]) -> Optional[Dict[int, int]]:
    mapping = {}
    for p in train_pairs:
        inp = np.asarray(p["input"])
        out = np.asarray(p["output"])
        if inp.shape != out.shape:
            return None
        diff_mask = inp != out
        if not np.any(diff_mask):
            continue
        unique_pairs = np.unique(np.stack([inp.ravel(), out.ravel()], axis=1), axis=0)
        for src, dst in unique_pairs:
            src, dst = int(src), int(dst)
            if src in mapping and mapping[src] != dst:
                return None
            mapping[src] = dst
    return mapping if mapping else None

# ----------------- Tiling & Scaling -----------------
def tile_grid(grid: np.ndarray, tr: int, tc: int) -> np.ndarray:
    return np.tile(grid, (tr, tc))

def scale_grid(grid: np.ndarray, factor: int) -> np.ndarray:
    return np.kron(grid, np.ones((factor, factor), dtype=int))

def infer_tiling_factors(train_pairs: List[Dict[str, Any]]) -> Optional[Tuple[int, int]]:
    ratios = []
    for p in train_pairs:
        hi, wi = get_shape(np.asarray(p["input"]))
        ho, wo = get_shape(np.asarray(p["output"]))
        if hi == 0 or wi == 0 or ho % hi != 0 or wo % wi != 0:
            return None
        ratios.append((ho // hi, wo // wi))
    if ratios and len(set(ratios)) == 1:
        return ratios[0]
    return None

def infer_scale_factor(train_pairs: List[Dict[str, Any]]) -> Optional[int]:
    scales = []
    for p in train_pairs:
        hi, wi = get_shape(np.asarray(p["input"]))
        ho, wo = get_shape(np.asarray(p["output"]))
        if hi == 0 or wi == 0 or ho % hi != 0 or wo % wi != 0:
            return None
        if (ho // hi) != (wo // wi):
            return None
        scales.append(ho // hi)
    if scales and len(set(scales)) == 1 and scales[0] > 1:
        return scales[0]
    return None

# ----------------- Gravity / Drop -----------------
def apply_gravity_down(grid: np.ndarray, bg: Optional[int] = None) -> np.ndarray:
    if bg is None:
        bg = get_most_common_color(grid)
    out = np.full_like(grid, bg)
    h, w = grid.shape
    for c in range(w):
        col_vals = [int(v) for v in grid[:, c] if v != bg]
        if col_vals:
            out[h - len(col_vals):h, c] = col_vals
    return out

def apply_gravity_up(grid: np.ndarray, bg: Optional[int] = None) -> np.ndarray:
    if bg is None:
        bg = get_most_common_color(grid)
    out = np.full_like(grid, bg)
    h, w = grid.shape
    for c in range(w):
        col_vals = [int(v) for v in grid[:, c] if v != bg]
        if col_vals:
            out[:len(col_vals), c] = col_vals
    return out

def apply_gravity_right(grid: np.ndarray, bg: Optional[int] = None) -> np.ndarray:
    if bg is None:
        bg = get_most_common_color(grid)
    out = np.full_like(grid, bg)
    h, w = grid.shape
    for r in range(h):
        row_vals = [int(v) for v in grid[r, :] if v != bg]
        if row_vals:
            out[r, w - len(row_vals):w] = row_vals
    return out

def apply_gravity_left(grid: np.ndarray, bg: Optional[int] = None) -> np.ndarray:
    if bg is None:
        bg = get_most_common_color(grid)
    out = np.full_like(grid, bg)
    h, w = grid.shape
    for r in range(h):
        row_vals = [int(v) for v in grid[r, :] if v != bg]
        if row_vals:
            out[r, :len(row_vals)] = row_vals
    return out

# ----------------- Summary / 1x1 Extraction -----------------
def extract_most_common_non_bg(grid: np.ndarray) -> np.ndarray:
    bg = get_most_common_color(grid)
    non_bg = grid[grid != bg]
    if len(non_bg) == 0:
        return np.array([[bg]], dtype=int)
    c = Counter(non_bg.tolist()).most_common(1)[0][0]
    return np.array([[c]], dtype=int)

def extract_least_common_non_bg(grid: np.ndarray) -> np.ndarray:
    bg = get_most_common_color(grid)
    non_bg = grid[grid != bg]
    if len(non_bg) == 0:
        return np.array([[bg]], dtype=int)
    c = Counter(non_bg.tolist()).most_common()[-1][0]
    return np.array([[c]], dtype=int)

def extract_component_count(grid: np.ndarray) -> np.ndarray:
    comps = get_connected_components(grid)
    return np.array([[min(9, len(comps))]], dtype=int)

def extract_unique_color_count(grid: np.ndarray) -> np.ndarray:
    bg = get_most_common_color(grid)
    unique_non_bg = len(set(grid.ravel()) - {bg})
    return np.array([[min(9, unique_non_bg)]], dtype=int)


# ----------------- Rule Candidate Class -----------------
class SymbolicRule:
    def __init__(self, name: str, fn: Callable[[np.ndarray], Optional[np.ndarray]], priority: int = 50):
        self.name = name
        self.fn = fn
        self.priority = priority

    def apply(self, grid: np.ndarray) -> Optional[np.ndarray]:
        try:
            res = self.fn(grid)
            if is_valid_grid(res):
                return res
            return None
        except Exception:
            return None

    def verifies_all(self, train_pairs: List[Dict[str, Any]]) -> bool:
        for p in train_pairs:
            inp = np.asarray(p["input"])
            expected = np.asarray(p["output"])
            pred = self.apply(inp)
            if pred is None or not np.array_equal(pred, expected):
                return False
        return True


def build_symbolic_rules(train_pairs: List[Dict[str, Any]]) -> List[SymbolicRule]:
    """Generates all deterministic/symbolic candidate transformations for a task."""
    rules = []

    # 1. Geometric Symmetries
    for name, fn in GEOMETRIC_TRANSFORMS:
        prio = 100 if name == "identity" else 75
        rules.append(SymbolicRule(name, fn, priority=prio))

    # 2. Crop Non-Background & Components
    rules.append(SymbolicRule("crop_non_bg", crop_non_bg, priority=80))
    rules.append(SymbolicRule("crop_largest_comp", crop_largest_component, priority=75))
    rules.append(SymbolicRule("crop_smallest_comp", crop_smallest_component, priority=70))
    rules.append(SymbolicRule("crop_remove_border", crop_remove_border, priority=65))
    rules.append(SymbolicRule("extract_outer_border", extract_outer_border, priority=60))

    # 3. Symmetry completions
    rules.append(SymbolicRule("complete_symmetry_h", complete_symmetry_h, priority=60))
    rules.append(SymbolicRule("complete_symmetry_v", complete_symmetry_v, priority=60))

    # 4. Crop non-bg + Symmetries
    for geo_name, geo_fn in GEOMETRIC_TRANSFORMS:
        if geo_name != "identity":
            rules.append(SymbolicRule(
                f"crop_non_bg_then_{geo_name}",
                lambda g, fn=geo_fn: fn(crop_non_bg(g)),
                priority=68
            ))

    # 5. Color Mapping
    color_map = infer_exact_color_map(train_pairs)
    if color_map is not None:
        rules.append(SymbolicRule(
            f"exact_color_map_{color_map}",
            lambda g, m=color_map: replace_colors(g, m),
            priority=90
        ))
        rules.append(SymbolicRule(
            f"crop_non_bg_then_color_map_{color_map}",
            lambda g, m=color_map: replace_colors(crop_non_bg(g), m),
            priority=85
        ))
        for geo_name, geo_fn in GEOMETRIC_TRANSFORMS:
            rules.append(SymbolicRule(
                f"{geo_name}_then_color_map_{color_map}",
                lambda g, fn=geo_fn, m=color_map: replace_colors(fn(g), m),
                priority=80
            ))

    # 6. Tiling
    tiling = infer_tiling_factors(train_pairs)
    if tiling is not None:
        tr, tc = tiling
        rules.append(SymbolicRule(f"tile_{tr}x{tc}", lambda g, tr=tr, tc=tc: tile_grid(g, tr, tc), priority=75))
        for geo_name, geo_fn in GEOMETRIC_TRANSFORMS:
            rules.append(SymbolicRule(
                f"{geo_name}_then_tile_{tr}x{tc}",
                lambda g, fn=geo_fn, tr=tr, tc=tc: tile_grid(fn(g), tr, tc),
                priority=70
            ))

    # 7. Scaling
    scale = infer_scale_factor(train_pairs)
    if scale is not None:
        rules.append(SymbolicRule(f"scale_{scale}x", lambda g, s=scale: scale_grid(g, s), priority=70))

    # 8. Gravity
    rules.append(SymbolicRule("gravity_down", apply_gravity_down, priority=50))
    rules.append(SymbolicRule("gravity_up", apply_gravity_up, priority=50))
    rules.append(SymbolicRule("gravity_right", apply_gravity_right, priority=50))
    rules.append(SymbolicRule("gravity_left", apply_gravity_left, priority=50))

    # 9. 1x1 Summaries
    rules.append(SymbolicRule("most_common_non_bg", extract_most_common_non_bg, priority=35))
    rules.append(SymbolicRule("least_common_non_bg", extract_least_common_non_bg, priority=35))
    rules.append(SymbolicRule("component_count", extract_component_count, priority=35))
    rules.append(SymbolicRule("unique_color_count", extract_unique_color_count, priority=35))

    return rules


def solve_task_symbolic(task: Dict[str, Any]) -> Dict[str, Any]:
    """
    Evaluates all symbolic rule candidates against the task demonstration pairs.
    Returns:
      {
        'has_exact': bool,
        'exact_rule_name': Optional[str],
        'exact_rule': Optional[SymbolicRule],
        'exact_predictions': List[np.ndarray],
        'top_symbolic_candidates': List[Tuple[str, np.ndarray, float]] # (name, grid, score)
      }
    """
    train_pairs = task.get("train", [])
    test_pairs = task.get("test", [])

    rules = build_symbolic_rules(train_pairs)
    exact_rules = []

    for rule in rules:
        if rule.verifies_all(train_pairs):
            exact_rules.append(rule)

    if exact_rules:
        # Sort by priority
        best_rule = max(exact_rules, key=lambda r: r.priority)
        preds = []
        for t in test_pairs:
            inp = np.asarray(t["input"])
            out = best_rule.apply(inp)
            preds.append(out if out is not None else inp.copy())
        return {
            "has_exact": True,
            "exact_rule_name": best_rule.name,
            "exact_rule": best_rule,
            "exact_predictions": preds,
            "top_symbolic_candidates": [(best_rule.name, p, 1000.0) for p in preds],
        }

    return {
        "has_exact": False,
        "exact_rule_name": None,
        "exact_rule": None,
        "exact_predictions": [],
        "top_symbolic_candidates": [],
    }


In [ ]:
%%writefile arc_loader.py
"""
arc_loader.py - Data serialization, augmentation, and prompt formatting for ARC-AGI-2
"""
import json
import numpy as np
from typing import List, Dict, Any, Optional

from arc_invariants import is_valid_grid

def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess) -> bool:
    return is_valid_grid(guess)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation) == list(range(10))
    a = np.asarray(a)
    if a.ndim == 3:
        if not invert:
            permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim == 2
        if invert:
            permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return "permute" + "".join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self) -> int:
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens: List[int], limit_rows: int = 30) -> Optional[np.ndarray]:
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except Exception:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None:
            return a
        for op in key.split(".")[1:]:
            if op == "rot90":
                a = np.rot90(a)
            elif op == "transpose":
                a = np.swapaxes(a, 0, 1)
            elif op.startswith("permute"):
                a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith("copy"):
                a = np.copy(a)
            elif op.startswith("out") or op.startswith("ex") or op.startswith("run"):
                a = a
            else:
                raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None:
            return a
        for op in key.split(".")[1:][::-1]:
            if op == "rot90":
                a = np.rot90(a, k=3)
            elif op == "transpose":
                a = np.swapaxes(a, 0, 1)
            elif op.startswith("permute"):
                a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith("copy"):
                a = np.copy(a)
            elif op.startswith("out") or op.startswith("ex") or op.startswith("run"):
                a = a
            else:
                raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies=None, keys=None, is_orig=False):
        replies = replies or {}
        if keys is not None:
            keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f:
            replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys if k in replies_parsed}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]["test"]))]
        return self.__class__(
            keys=[f"{k}_{i}" for k, i in key_indices],
            queries={f"{k}_{i}": {"train": self.queries[k]["train"], "test": [self.queries[k]["test"][i]]} for k, i in key_indices},
            replies={f"{k}_{i}": [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys=[k for d in datasets for k in d.keys],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys = []
        for k0 in self.keys:
            desc = (("copy{i}" if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t == "input" or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None:
            stack = mod_func.__name__.startswith("rot")
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]["train"])
        query = formatter.fmt_query(self.queries[key]["test"])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ""
        text = train + query + reply if reply else formatter.fmt_train(self.queries[key]["train"], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train + query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if name == "input":
                return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name == "reply":
                return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else:
                assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None:
                    self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len < temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split(".")[-1].startswith("ex"):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split(".")
                assert key_split[-1].startswith("ex")
                key = ".".join(key_split[:-1] + [f"ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}"])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k == "train" else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None:
                new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)

    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]["train"])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None:
                p = p[:keep_max]
            new_key = f"{key}.ex" + ("-" if (p.max() > 9) else "").join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k == "train" else v) for k, v in self.queries[key].items()}
            if key in self.replies:
                new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig == True, "Must be run on original dataset."
        submission = {k: [{f"attempt_{i+1}": [[0]] for i in range(2)} for _ in range(len(self.queries[k]["test"]))] for k in self.keys}
        if results is not None:
            self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f"*** Generating submission for {len(results)} outputs...")
        for k, v in results.items():
            base_id, base_nr = k.split("_")
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f"attempt_{i+1}"] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig == True, "Must be run on original dataset."
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ["attempt_1", "attempt_2"]:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score


In [ ]:
%%writefile arc_decoder.py
"""
arc_decoder.py - Advanced V2 Candidate Scoring, Demonstration Verification, and Attempt Diversification
"""
import os
import bz2
import pickle
import numpy as np
from typing import List, Dict, Any, Tuple, Optional

from arc_invariants import (
    is_valid_grid,
    validate_candidate_invariants,
    infer_shape_rule,
    grid_hamming_distance,
)

def hashable(guess) -> Tuple[Tuple[int, ...], ...]:
    return tuple(map(tuple, guess))

def _valid_sample(sample: Dict[str, Any]) -> bool:
    try:
        sol = np.asarray(sample["solution"])
        if not is_valid_grid(sol):
            return False
        if not np.isfinite(sample.get("beam_score", 0.0)):
            return False
        score_aug = sample.get("score_aug", [])
        if len(score_aug) and not np.all(np.isfinite(score_aug)):
            return False
        return True
    except Exception:
        return False

# ----------------- Scoring Algorithms -----------------

def score_kgmon(guesses: Dict[str, Any]) -> List[np.ndarray]:
    """Baseline kgmon consensus scoring: Vote Count - Mean Augmented NLL."""
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores.setdefault(h, [[], g["solution"]])
        x[0].append(g)

    ranked = []
    for sc_list, sol in scores.values():
        inf_score = len(sc_list)
        aug_nlls = [np.mean(g["score_aug"]) for g in sc_list if len(g.get("score_aug", []))]
        mean_aug = float(np.mean(aug_nlls)) if aug_nlls else 0.0
        score = inf_score - mean_aug
        ranked.append((score, sol))

    ranked.sort(key=lambda x: x[0], reverse=True)
    return [x[1] for x in ranked]


def score_v2(guesses: Dict[str, Any], task_train: Optional[List[Dict[str, Any]]] = None, test_input: Optional[np.ndarray] = None) -> List[np.ndarray]:
    """
    V2 Enhanced Consensus Scoring:
    - Vote Count - Mean Augmented NLL
    - Symbolic exact rule bonus (+1000.0)
    - Structural invariant penalties (shape mismatch & color hallucination)
    """
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores.setdefault(h, [[], g["solution"], g.get("is_symbolic_exact", False)])
        x[0].append(g)
        if g.get("is_symbolic_exact", False):
            x[2] = True

    shape_rule = infer_shape_rule(task_train) if task_train else None

    ranked = []
    for sc_list, sol, is_exact in scores.values():
        sol_arr = np.asarray(sol)
        inf_score = len(sc_list)
        aug_nlls = [np.mean(g["score_aug"]) for g in sc_list if len(g.get("score_aug", []))]
        mean_aug = float(np.mean(aug_nlls)) if aug_nlls else 0.0
        base_score = inf_score - mean_aug

        # Exact verified symbolic rule bonus
        if is_exact:
            base_score += 1000.0

        # Structural invariant check
        if task_train and test_input is not None:
            is_plausible, penalty = validate_candidate_invariants(sol_arr, task_train, test_input, shape_rule)
            base_score -= penalty

        ranked.append((base_score, sol_arr))

    ranked.sort(key=lambda x: x[0], reverse=True)
    return [x[1] for x in ranked]


def select_orthogonal_top2(
    ordered_candidates: List[np.ndarray],
    fallback_input: Optional[np.ndarray] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Ensures attempt_1 and attempt_2 represent diverse hypotheses.
    Attempt 1: Best overall candidate.
    Attempt 2: Best alternative candidate with non-zero structural distance.
    """
    if not ordered_candidates:
        fb = fallback_input if fallback_input is not None else np.array([[0]], dtype=int)
        return fb, np.array([[0]], dtype=int)

    attempt_1 = ordered_candidates[0]
    attempt_2 = None

    for cand in ordered_candidates[1:]:
        if not np.array_equal(cand, attempt_1):
            attempt_2 = cand
            break

    if attempt_2 is None:
        attempt_2 = np.array([[0]], dtype=int)

    return attempt_1, attempt_2


class ArcDecoder:

    def __init__(self, dataset, n_guesses: int = 2):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}
        try:
            self.valid_keys = set(dataset.keys)
        except Exception:
            self.valid_keys = None

    def load_decoded_results(self, store: str, run_name: str = "") -> int:
        if not os.path.isdir(store):
            print(f"*** No decoded results at {store}")
            return 0
        n_files = n_samples = n_bad = 0
        for key in os.listdir(store):
            if key.startswith(".") or key.endswith((".json", ".txt", ".tmp")):
                continue
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as e:
                print(f"*** Skipping corrupt shard {key}: {e}")
                n_bad += 1
                continue
            n_files += 1
            base_key = key.split(".")[0]
            if self.valid_keys is not None and base_key not in self.valid_keys:
                n_bad += 1
                continue
            for i, sample in enumerate(outputs):
                if not _valid_sample(sample):
                    n_bad += 1
                    continue
                self.decoded_results.setdefault(base_key, {})[f"{key}{run_name}.out{i}"] = sample
                n_samples += 1
        print(f"*** Loaded {n_files} shards / {n_samples} samples from {store} (skipped {n_bad})")
        return n_samples

    def inject_symbolic_results(self, symbolic_dict: Dict[str, np.ndarray]):
        """Injects exact verified symbolic candidate grids directly into decoded pool."""
        count = 0
        for base_key, pred_grid in symbolic_dict.items():
            if not is_valid_grid(pred_grid):
                continue
            sample = {
                "beam_score": 0.0,
                "score_aug": [0.0] * 8,
                "solution": np.asarray(pred_grid, dtype=int),
                "is_symbolic_exact": True,
            }
            self.decoded_results.setdefault(base_key, {})["symbolic_exact.out0"] = sample
            count += 1
        if count > 0:
            print(f"*** Injected {count} exact symbolic predictions into decoder pool")

    def candidate_stats(self) -> Dict[str, Dict[str, int]]:
        """Per output key: number of unique candidate grids and number of samples."""
        stats = {}
        for bk, v in self.decoded_results.items():
            uniq = {hashable(g["solution"]) for g in v.values()}
            stats[bk] = dict(unique=len(uniq), samples=len(v))
        return stats

    def run_selection_algo(self, selection_algorithm=score_v2) -> Dict[str, List[np.ndarray]]:
        results = {}
        for bk, v in self.decoded_results.items():
            task_train = None
            test_input = None
            try:
                task_id = bk.split("_")[0]
                task_data = self.dataset.queries.get(bk, self.dataset.queries.get(task_id))
                if task_data:
                    task_train = task_data.get("train")
                    test_input = np.asarray(task_data["test"][0]["input"])
            except Exception:
                pass

            if selection_algorithm == score_v2:
                ordered = score_v2({k: g for k, g in v.items()}, task_train=task_train, test_input=test_input)
            else:
                ordered = selection_algorithm({k: g for k, g in v.items()})

            results[bk] = ordered
        return results

    def get_diverse_attempts(self, selection_algorithm=score_v2) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
        selected = self.run_selection_algo(selection_algorithm)
        attempts = {}
        for bk, cand_list in selected.items():
            fb = None
            try:
                task_id = bk.split("_")[0]
                task_data = self.dataset.queries.get(bk, self.dataset.queries.get(task_id))
                if task_data:
                    fb = np.asarray(task_data["test"][0]["input"])
            except Exception:
                pass
            att1, att2 = select_orthogonal_top2(cand_list, fallback_input=fb)
            attempts[bk] = (att1, att2)
        return attempts


In [ ]:
%%writefile arc_solver.py
"""
arc_solver.py - Test-Time Training (LoRA) & Constrained Turbo DFS Tree Search Engine for ARC-AGI-2
"""
import sys
import os
import glob
# Fix6: reduce fragmentation — must be set before CUDA context init
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

sys.setrecursionlimit(5000)

import site
# Prioritize real system site-packages so real PyTorch is loaded
for sp in reversed(site.getsitepackages()):
    if sp in sys.path:
        sys.path.remove(sp)
    sys.path.insert(0, sp)

# Remove any broken torchao from sys.modules
if "torchao" in sys.modules:
    del sys.modules["torchao"]

# Patch peft to safely treat torchao as unavailable
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

# Move any utility script shadow folders to the end of sys.path
for p in list(sys.path):
    if "pip_install_unsloth_flash_patch" in p or "usr/lib/notebooks" in p:
        sys.path.remove(p)
        sys.path.append(p)

# Discover unsloth from Kaggle utility scripts or input datasets and append to sys.path
_unsloth_search_paths = (
    glob.glob("/kaggle/usr/lib/notebooks/*/pip_install_unsloth*") +
    glob.glob("/kaggle/usr/lib/notebooks/*/*/pip_install_unsloth*") +
    glob.glob("/kaggle/usr/lib/**", recursive=True) +
    glob.glob("/kaggle/input/**/unsloth", recursive=True)
)
for p in _unsloth_search_paths:
    if os.path.isdir(p):
        if os.path.isfile(os.path.join(p, "unsloth", "__init__.py")):
            if p not in sys.path:
                sys.path.append(p)
        elif os.path.basename(p) == "unsloth" and os.path.isfile(os.path.join(p, "__init__.py")):
            parent = os.path.dirname(p)
            if parent not in sys.path:
                sys.path.append(parent)

try:
    from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
    HAS_UNSLOTH = True
except ImportError:
    HAS_UNSLOTH = False
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
    from peft import LoraConfig, get_peft_model
    UnslothTrainingArguments = TrainingArguments
    UnslothTrainer = Trainer

from arc_loader import ArcDataset, QwenFormatter, is_valid_solution
from arc_decoder import hashable
from arc_invariants import is_valid_grid

import gc
import io
import time
import zlib
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict
from typing import Any, Union, List, Dict, Tuple, Optional
from transformers import DataCollatorForLanguageModeling, TrainerCallback
import logging
from contextlib import redirect_stdout, redirect_stderr
from peft import get_peft_model_state_dict, set_peft_model_state_dict
import bz2
import pickle
import traceback

logging.disable(logging.WARNING)

def _env(name, default, cast):
    v = os.getenv(name)
    return cast(v) if v not in (None, "") else default

CFG = dict(
    model_path      = _env("ARC_MODEL_PATH", "", str),
    out_dir         = _env("ARC_OUT_DIR", "/kaggle/inference_outputs", str),
    lora_seed       = _env("ARC_LORA_SEED", 42, int),
    train_aug_seed  = _env("ARC_TRAIN_AUG_SEED", 1, int),
    n_train_aug     = _env("ARC_N_TRAIN_AUG", 16, int),     # x8 geometries = 128 sequences
    num_epochs      = _env("ARC_EPOCHS", 1, int),
    learning_rate   = _env("ARC_LR", 5e-5, float),
    eval_aug_seed   = _env("ARC_EVAL_AUG_SEED", 2, int),
    n_eval_aug      = _env("ARC_N_EVAL_AUG", 2, int),       # x8 geometries = 16 decoded views
    min_prob        = _env("ARC_MIN_PROB", 0.2, float),     # DFS cumulative prob threshold
    dfs_window      = _env("ARC_DFS_WINDOW", 540.0, float), # seconds per DFS call
    task_cap        = _env("ARC_TASK_CAP", 1200.0, float),  # seconds per task (decode phase)
    score_seed_off  = _env("ARC_SCORE_SEED_OFFSET", 0, int),
    decode_batch    = _env("ARC_DECODE_BATCH", 4, int),
    early_stop_loss = _env("ARC_EARLY_STOP_LOSS", 5e-4, float), # TTT loss threshold for early stop
)

ARC_VOCAB = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "Ċ": 10, "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


def resolve_model_dir():
    if CFG["model_path"] and os.path.isdir(CFG["model_path"]):
        return CFG["model_path"]
    candidates = [
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
        "/kaggle/input/arc-qwen-model",
        "/kaggle/input/qwen-models",
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, "config.json")):
            return c
    for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(c)
        if os.path.isfile(os.path.join(d, "tokenizer.json")) or os.path.isfile(os.path.join(d, "tokenizer_config.json")):
            return d
    return "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"


def stable_seed(key, offset=0):
    return (zlib.crc32(key.encode("utf-8")) + offset) % (1024 ** 2)


def make_training_args(**kwargs):
    if HAS_UNSLOTH:
        return UnslothTrainingArguments(**kwargs)
    import inspect
    sig = inspect.signature(TrainingArguments.__init__)
    params = sig.parameters
    if any(param.kind == inspect.Parameter.VAR_KEYWORD for param in params.values()):
        return TrainingArguments(**kwargs)
    cleaned = {k: v for k, v in kwargs.items() if k in params}
    return TrainingArguments(**cleaned)


class EarlyStoppingOnLossCallback(TrainerCallback):
    """Stops TTT fine-tuning early if demonstration loss has converged near zero."""
    def __init__(self, threshold=5e-4):
        self.threshold = threshold

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            if logs["loss"] < self.threshold:
                control.should_training_stop = True


class UnslothFixedTrainer(UnslothTrainer):

    def __init__(self, *args, **kwargs):
        import inspect
        sig = inspect.signature(UnslothTrainer.__init__)
        params = sig.parameters
        cleaned = {}
        for k, v in kwargs.items():
            if k in params:
                cleaned[k] = v
            elif k == "tokenizer" and "processing_class" in params:
                cleaned["processing_class"] = v
            elif k == "processing_class" and "tokenizer" in params:
                cleaned["tokenizer"] = v
        super().__init__(*args, **cleaned)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        import torch.nn as nn
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if getattr(self.args, "past_index", -1) >= 0:
            self._past = outputs[self.args.past_index]

        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss_fct = nn.CrossEntropyLoss()
                loss = loss_fct(outputs.logits.view(-1, outputs.logits.shape[-1]), labels.view(-1))
            else:
                loss = self.label_smoother(outputs, labels, shift_labels=True)
        else:
            if isinstance(outputs, dict) and "loss" not in outputs:
                raise ValueError("The model did not return a loss from inputs.")
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]

        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            labels_np = labels.detach().cpu().numpy()
            user_start_idx = np.where(labels_np == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels_np == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels_np == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


_ARC_TOKEN_ID_CACHE = {}

def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time, dfs_window) -> dict:
    n = logits.size(0)
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)
    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x: x[0])

    while time.time() - start_time < dfs_window and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos + 1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
            dfs_window=dfs_window,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))
        # Fix4: free per-iteration KV and logits to avoid recursive cache blowup
        try:
            del outputs, next_suffixes, batch_tokens, batch_scores
        except Exception:
            pass

    # free temporary NLL tensors
    try:
        del logits_f, arc_logits, nll, token_ids
    except Exception:
        pass
    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time, dfs_window):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    # cache reference kept for turbo_dfs, free logits after
    _logits_slice = outputs.logits[:, -1].detach()
    _past = outputs.past_key_values
    # free full outputs logits early to save memory
    try:
        del outputs
    except Exception:
        pass
    suffixes = turbo_dfs(
        model,
        logits=_logits_slice,
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=_past,
        start_time=time.time(),
        end_time=end_time,
        dfs_window=dfs_window,
    )
    try:
        del _logits_slice, _past, input_ids
    except Exception:
        pass
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x: x[0])
        result.append((batch_id, sorted_beams))
    try:
        del suffixes
    except Exception:
        pass
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    # free outputs early — keep only float logits
    try:
        del outputs, padded_tokens
    except Exception:
        pass
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
        try:
            del positions, target_tokens, answer_log_probs
        except Exception:
            pass
    # Fix5: free large logits tensors before return
    try:
        del input_ids, batch_logits, batch_log_norm, batch_query_tokens, batch_answer_tokens, batch_tokens, batch_lengths
    except Exception:
        pass
    return result


def make_view_batches(eval_ds, n_perm, batch_size):
    test_id_to_subkeys = defaultdict(list)
    for subkey in sorted(eval_ds.keys):
        test_id = subkey.split(".")[0].split("_")[1]
        test_id_to_subkeys[test_id].append(subkey)
    groups_a = [0, 2, 1, 3]
    groups_b = [4, 6, 5, 7]
    batches = []
    for geos in (groups_a, groups_b):
        for test_id, subkeys in test_id_to_subkeys.items():
            if n_perm == 2 and batch_size == 4:
                for a, b in ((geos[0], geos[1]), (geos[2], geos[3])):
                    batches.append(subkeys[a * n_perm:(a + 1) * n_perm] + subkeys[b * n_perm:(b + 1) * n_perm])
            else:
                views = []
                for g in geos:
                    views.extend(subkeys[g * n_perm:(g + 1) * n_perm])
                for i in range(0, len(views), batch_size):
                    batches.append(views[i:i + batch_size])
    return batches


def worker(rank, queue, end_time, test_path=None):
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=_env("ARC_LORA_R", 64, int),
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=_env("ARC_GRAD_CKPT", 1, int) == 1,
        random_state=CFG["lora_seed"],
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=CFG["num_epochs"],
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=CFG["learning_rate"],
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=CFG["lora_seed"],
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="steps",
        logging_steps=16,
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=_env("ARC_GRAD_CKPT", 1, int) == 1,
        remove_unused_columns=False,
    )

    max_seq_length = 8192

    model_dir = resolve_model_dir()
    print(f"[Rank {rank}] model dir: {model_dir}")
    print(f"[Rank {rank}] config: {CFG}")

    if HAS_UNSLOTH:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_dir,
            full_finetuning=False,
            load_in_4bit=False,
            local_files_only=True,
            use_gradient_checkpointing=_env("ARC_GRAD_CKPT", 1, int) == 1,
            max_seq_length=max_seq_length,
        )
        model = FastLanguageModel.get_peft_model(model, **peft_params)
    else:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        from peft import LoraConfig, get_peft_model
        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.bfloat16, local_files_only=True)
        peft_config = LoraConfig(
            r=peft_params.get("r", 64),
            lora_alpha=peft_params.get("lora_alpha", 32),
            target_modules=peft_params.get("target_modules", ["q_proj", "k_proj", "v_proj", "o_proj"]),
            lora_dropout=peft_params.get("lora_dropout", 0.0),
            bias="none",
            task_type="CAUSAL_LM"
        )
        model = get_peft_model(model, peft_config)
        model = model.to("cuda")
        if _env("ARC_GRAD_CKPT", 1, int) == 1:
            try:
                model.gradient_checkpointing_enable()
                model.enable_input_require_grads()
            except Exception:
                pass

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    # Fix2: keep LoRA snapshot on CPU to free ~1-2GB GPU across tasks
    default_weights = {k: v.clone().detach().cpu() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)
    max_new_tokens = formatter.max_new_tokens()
    max_score = -np.log(CFG["min_prob"])

    if test_path is None:
        candidates_test = [
            "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json",
            "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json",
            "/kaggle/input/arc-prize-2026/arc-agi_test_challenges.json",
            "/kaggle/input/arc-agi_test_challenges.json",
            "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc_agi_test_challenges.json",
            "/kaggle/input/arc-prize-2026-arc-agi-2/arc_agi_test_challenges.json",
            "/kaggle/input/arc-prize-2026/arc_agi_test_challenges.json",
        ]
        candidates_eval = [
            "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json",
            "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json",
            "/kaggle/input/arc-prize-2026/arc-agi_evaluation_challenges.json",
            "/kaggle/input/arc-agi_evaluation_challenges.json",
            "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc_agi_evaluation_challenges.json",
            "/kaggle/input/arc-prize-2026-arc-agi-2/arc_agi_evaluation_challenges.json",
            "/kaggle/input/arc-prize-2026/arc_agi_evaluation_challenges.json",
        ]
        resolved_test = next((c for c in candidates_test if os.path.exists(c)), None)
        if resolved_test is None:
            matches = glob.glob("/kaggle/input/**/arc*test_challenges.json", recursive=True)
            if matches:
                resolved_test = matches[0]

        if rerun_mode or resolved_test is not None:
            test_path = resolved_test or candidates_test[0]
        else:
            resolved_eval = next((c for c in candidates_eval if os.path.exists(c)), None)
            if resolved_eval is None:
                matches = glob.glob("/kaggle/input/**/arc*evaluation_challenges.json", recursive=True)
                if matches:
                    resolved_eval = matches[0]
            test_path = resolved_eval or candidates_eval[0]

    arc_test_set = ArcDataset.from_file(test_path)
    dir_outputs = CFG["out_dir"]
    os.makedirs(dir_outputs, exist_ok=True)

    early_stop_cb = EarlyStoppingOnLossCallback(threshold=CFG["early_stop_loss"])

    while True:
        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        start_time = time.time()

        try:
            torch.cuda.reset_peak_memory_stats()

            # Fix2: restore from CPU snapshot, moving to model device
            _dw_on_device = {k: v.to(model.device) for k, v in default_weights.items()}
            set_peft_model_state_dict(
                model,
                _dw_on_device,
                adapter_name="default",
            )
            del _dw_on_device

            if HAS_UNSLOTH:
                model = FastLanguageModel.for_training(model)

            puzzle_ds = arc_test_set.change_keys([key])
            train_ds = puzzle_ds.augment(n=CFG["n_train_aug"], shfl_keys=True, seed=CFG["train_aug_seed"])
            train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

            train_items = []
            for item in train_ds.as_list(formatter):
                tokens = tokenizer.encode(item["text"])
                train_items.append({"input_ids": tokens})

            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_items),
                    args=make_training_args(**train_args),
                    callbacks=[early_stop_cb],
                )
                stats = trainer.train()
                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
                del trainer

            if HAS_UNSLOTH:
                model = FastLanguageModel.for_inference(model)

            try:
                del trainer  # already deleted above, ensure freed
            except Exception:
                pass
            gc.collect()
            try:
                torch.cuda.synchronize()
            except Exception:
                pass
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass

            memory_allocated = torch.cuda.max_memory_allocated() // 1024 ** 2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for training")
            torch.cuda.reset_peak_memory_stats()
            print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")
            # also free train artifacts after stats logged — keep puzzle_ds for eval
            for _n in ["train_items", "train_ds"]:
                try:
                    # manual del via exec to avoid locals() pitfall
                    if _n == "train_items":
                        del train_items
                    elif _n == "train_ds":
                        del train_ds
                except Exception:
                    pass
            gc.collect()
            try:
                torch.cuda.synchronize()
            except Exception:
                pass
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass

            puzzle_ds_multi = puzzle_ds.split_multi_replies()
            eval_ds = puzzle_ds_multi.augment(n=CFG["n_eval_aug"], seed=CFG["eval_aug_seed"])
            eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)

            batches = make_view_batches(eval_ds, CFG["n_eval_aug"], CFG["decode_batch"])

            with torch.inference_mode():
                known_scores = {}
                for subkeys in batches:
                    spend_time = time.time() - start_time
                    if spend_time > CFG["task_cap"] or time.time() > end_time:
                        print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                        break

                    print(f"[Rank {rank}] decoding {subkeys}")
                    tokens = []
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        tokens.append(tokenizer.encode(data["input"]))

                    dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time, CFG["dfs_window"])

                    for subkey_id, scored_beams in dfs_result:
                        subkey = subkeys[subkey_id]
                        bk = subkey.split(".")[0]
                        decoded_result = []

                        for beam_score, tokens_ in scored_beams:
                            array = formatter.convert_tokens_to_array(tokens_)
                            if array is None:
                                continue

                            solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)
                            if not is_valid_grid(solution):
                                continue

                            grid_id = (bk, tuple(map(tuple, solution)))

                            if grid_id in known_scores:
                                augmented_scores = known_scores[grid_id]
                            else:
                                aug_dataset = ArcDataset(
                                    keys=[bk],
                                    queries={bk: puzzle_ds_multi.queries.get(bk)},
                                    replies={bk: [solution.tolist()]},
                                )
                                aug_dataset = aug_dataset.augment(seed=stable_seed(bk, CFG["score_seed_off"]))
                                aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)
                                aug_queries = []
                                aug_answers = []
                                for augmented_sample in aug_dataset.as_list(formatter):
                                    aug_queries.append(augmented_sample["input"])
                                    aug_answers.append(augmented_sample["reply"])
                                augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                                augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                                augmented_scores = augmented_scores1 + augmented_scores2
                                known_scores[grid_id] = augmented_scores
                                # Fix5: free augmentation artifacts immediately
                                try:
                                    del aug_dataset, aug_queries, aug_answers, augmented_scores1, augmented_scores2
                                except Exception:
                                    pass

                            decoded_result.append({
                                "beam_score": beam_score,
                                "score_aug": augmented_scores,
                                "solution": solution,
                            })

                        if len(decoded_result):
                            shard_path = os.path.join(dir_outputs, subkey)
                            tmp_path = shard_path + f".tmp.{rank}.{os.getpid()}"
                            with bz2.BZ2File(tmp_path, "w") as f:
                                pickle.dump(decoded_result, f)
                            os.replace(tmp_path, shard_path)
                        try:
                            del decoded_result, scored_beams
                        except Exception:
                            pass
                    # per-batch cleanup
                    try:
                        del dfs_result, tokens
                    except Exception:
                        pass
                    gc.collect()

            memory_allocated = torch.cuda.max_memory_allocated() // 1024 ** 2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
            # --- Fix1: symmetric hygiene — free inference artifacts before next task ---
            try:
                del puzzle_ds
            except Exception:
                pass
            try:
                del train_ds
            except Exception:
                pass
            try:
                del puzzle_ds_multi
            except Exception:
                pass
            try:
                del eval_ds
            except Exception:
                pass
            try:
                del batches
            except Exception:
                pass
            try:
                del known_scores
            except Exception:
                pass
            try:
                del dfs_result
            except Exception:
                pass
            try:
                del tokens
            except Exception:
                pass
            gc.collect()
            try:
                torch.cuda.synchronize()
            except Exception:
                pass
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass
            torch.cuda.reset_peak_memory_stats()

        except Exception as e:
            print(f"[Rank {rank}] ERROR on puzzle {key}: {type(e).__name__}: {e}")
            traceback.print_exc()
            if HAS_UNSLOTH:
                try:
                    model = FastLanguageModel.for_inference(model)
                except Exception:
                    pass
            for _n2 in ["puzzle_ds", "train_ds", "train_items", "puzzle_ds_multi", "eval_ds", "batches", "known_scores", "dfs_result", "tokens", "aug_dataset", "aug_queries", "aug_answers"]:
                try:
                    if _n2 == "puzzle_ds":
                        del puzzle_ds
                    elif _n2 == "train_ds":
                        del train_ds
                    elif _n2 == "train_items":
                        del train_items
                    elif _n2 == "puzzle_ds_multi":
                        del puzzle_ds_multi
                    elif _n2 == "eval_ds":
                        del eval_ds
                    elif _n2 == "batches":
                        del batches
                    elif _n2 == "known_scores":
                        del known_scores
                    elif _n2 == "dfs_result":
                        del dfs_result
                    elif _n2 == "tokens":
                        del tokens
                    elif _n2 == "aug_dataset":
                        del aug_dataset
                    elif _n2 == "aug_queries":
                        del aug_queries
                    elif _n2 == "aug_answers":
                        del aug_answers
                except Exception:
                    pass
            gc.collect()
            try:
                torch.cuda.synchronize()
            except Exception:
                pass
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass
            torch.cuda.reset_peak_memory_stats()
            if isinstance(e, torch.cuda.OutOfMemoryError):
                torch.cuda.synchronize()

        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
"""
starter.py - Multi-GPU Orchestrator with Integrated Fast Symbolic Pre-Pass
"""
import os
# Fix6: must be set before torch CUDA init
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
import sys
import time
import json
try:
    import torch
    import torch.multiprocessing as mp
except ImportError:
    torch = None
    import multiprocessing as mp

import argparse
import traceback
import bz2
import pickle
import numpy as np

from arc_symbolic import solve_task_symbolic
from arc_invariants import is_valid_grid


def local_worker(rank, queue, end_time, test_path, marker_dir):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    torch.set_default_device("cpu")

    # Stagger worker imports to prevent simultaneous JIT compilation collisions
    if rank > 0:
        waited = 0
        while not os.path.exists(os.path.join(marker_dir, f"worker{rank-1}")) and waited < 900:
            time.sleep(5)
            waited += 5

    from arc_solver import worker

    with open(os.path.join(marker_dir, f"worker{rank}"), "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")

    attempts = 0
    while attempts < 2 and time.time() < end_time:
        attempts += 1
        try:
            worker(rank, queue, end_time, test_path=test_path)
            break
        except Exception as e:
            print(f"[Rank {rank}] worker crashed ({type(e).__name__}: {e}); attempt {attempts}")
            traceback.print_exc()
            try:
                import gc
                gc.collect()
                torch.cuda.empty_cache()
            except Exception:
                pass
            if attempts >= 2:
                print(f"[Rank {rank}] giving up.")

    print(f"[Rank {rank}] done!")


def estimated_work(task):
    """Token cost proxy for ordering tasks cheap-first."""
    def ntok(g):
        return len(g) * (len(g[0]) + 1)
    train_tokens = sum(ntok(p["input"]) + ntok(p["output"]) for p in task["train"])
    ratios = [ntok(p["output"]) / max(1, ntok(p["input"])) for p in task["train"]]
    ratios.sort()
    ratio = ratios[len(ratios) // 2]
    test_tokens = sum(ntok(t["input"]) * (1 + ratio) for t in task["test"])
    return train_tokens * 16 + test_tokens * 8 * len(task["test"])


def run_symbolic_prepass(data, keys, symbolic_out_dir="/kaggle/working/symbolic_outputs"):
    """
    Executes fast symbolic pre-pass on CPU.
    Returns:
      solved_tasks: list of task keys solved with 100% exact demonstration verification.
    """
    os.makedirs(symbolic_out_dir, exist_ok=True)
    solved_tasks = []
    symbolic_summary = {}

    t0 = time.time()
    for k in keys:
        task = data[k]
        res = solve_task_symbolic(task)
        if res["has_exact"]:
            solved_tasks.append(k)
            preds = res["exact_predictions"]
            symbolic_summary[k] = {
                "rule": res["exact_rule_name"],
                "predictions": [p.tolist() if isinstance(p, np.ndarray) else p for p in preds]
            }

            # Save per-test-item candidate
            for i, p in enumerate(preds):
                subkey = f"{k}_{i}"
                sample = {
                    "beam_score": 0.0,
                    "score_aug": [0.0] * 8,
                    "solution": np.asarray(p, dtype=int),
                    "is_symbolic_exact": True,
                }
                out_path = os.path.join(symbolic_out_dir, f"{subkey}.symbolic")
                with bz2.BZ2File(out_path, "w") as f:
                    pickle.dump([sample], f)

    elapsed = time.time() - t0
    print(f"[Symbolic Pre-Pass] checked {len(keys)} tasks in {elapsed:.2f}s: {len(solved_tasks)} solved with exact verified rules!")
    summary_path = os.path.join(os.path.dirname(symbolic_out_dir), "symbolic_summary.json")
    with open(summary_path, "w") as f:
        json.dump(symbolic_summary, f, indent=2)

    return solved_tasks


def resolve_challenges_path(rerun=False):
    fname = "arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json"
    alt_fname = "arc_agi_test_challenges.json" if rerun else "arc_agi_evaluation_challenges.json"
    candidates = [
        f"/kaggle/input/competitions/arc-prize-2026-arc-agi-2/{fname}",
        f"/kaggle/input/arc-prize-2026-arc-agi-2/{fname}",
        f"/kaggle/input/arc-prize-2026/{fname}",
        f"/kaggle/input/{fname}",
        f"/kaggle/input/competitions/arc-prize-2026-arc-agi-2/{alt_fname}",
        f"/kaggle/input/arc-prize-2026-arc-agi-2/{alt_fname}",
        f"/kaggle/input/arc-prize-2026/{alt_fname}",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    import glob
    for pattern in [f"/kaggle/input/**/{fname}", f"/kaggle/input/**/{alt_fname}"]:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]
    return f"/kaggle/input/competitions/arc-prize-2026-arc-agi-2/{fname}"


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    parser.add_argument("--keys-file", type=str, default="")
    parser.add_argument("--nprocs", type=int, default=0)
    parser.add_argument("--order", type=str, default="cheap", choices=["cheap", "sorted", "file"])
    parser.add_argument("--test-path", type=str, default="")
    parser.add_argument("--marker-dir", type=str, default="/kaggle/working/markers")
    parser.add_argument("--skip-symbolic", action="store_true", default=False)
    args, _ = parser.parse_known_args()

    rerun_mode = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

    if args.test_path:
        test_path = args.test_path
    else:
        test_file = resolve_challenges_path(rerun=True)
        if rerun_mode or os.path.exists(test_file):
            test_path = test_file
        else:
            test_path = resolve_challenges_path(rerun=False)

    with open(test_path, "r") as f:
        data = json.load(f)

    if args.keys_file:
        with open(args.keys_file) as f:
            keys = [k for k in json.load(f) if k in data]
    else:
        keys = sorted(data.keys())
        test_file_exists = os.path.exists(resolve_challenges_path(rerun=True))
        if not rerun_mode and not test_file_exists:
            debug_keys = os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")
            keys = [k for k in keys if k in debug_keys]

    # 1. Run Symbolic Pre-Pass
    if not args.skip_symbolic:
        solved_tasks = run_symbolic_prepass(data, keys)
        if solved_tasks:
            orig_len = len(keys)
            keys = [k for k in keys if k not in set(solved_tasks)]
            print(f"[starter] Excluded {len(solved_tasks)} symbolically solved tasks from GPU queue ({orig_len} -> {len(keys)} remaining)")

    # 2. Sort remaining tasks
    if args.order == "cheap":
        keys = sorted(keys, key=lambda k: estimated_work(data[k]))
    elif args.order == "sorted":
        keys = sorted(keys)

    nprocs = args.nprocs or min(4, max(1, torch.cuda.device_count()))
    os.makedirs(args.marker_dir, exist_ok=True)
    for f_ in os.listdir(args.marker_dir):
        try:
            os.remove(os.path.join(args.marker_dir, f_))
        except Exception:
            pass

    print(f"[starter] {len(keys)} tasks, {nprocs} workers, order={args.order}, "
          f"budget={(args.end_time - time.time())/60:.1f} min, test_path={test_path}")

    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    try:
        mp.spawn(local_worker, args=(queue, args.end_time, test_path, args.marker_dir), nprocs=nprocs)
    except Exception as e:
        print(f"[starter] spawn finished with error: {type(e).__name__}: {e}")
        traceback.print_exc()
    print("[starter] finished.")


In [ ]:
# ---------------------------------------------------------------------------
# Phase 1 — Primary pass: Fast Symbolic Pre-Pass on CPU + Multi-GPU LoRA TTT
# with cheap-first task scheduling and early stopping.
# ---------------------------------------------------------------------------
import subprocess, sys, time, os
os.environ.update({
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
    "OMP_NUM_THREADS": "12",
    "PYTHONHASHSEED": "0",
    "ARC_OUT_DIR": "/kaggle/inference_outputs",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "PYTORCH_ALLOC_CONF": "expandable_segments:True",
    "ARC_LORA_R": "64",
    "ARC_GRAD_CKPT": "1",
})
phase1_start = time.time()
rc = subprocess.call([sys.executable, "starter.py", "--end-time", f"{global_end_time}", "--order", "cheap"])
print(f"phase-1 rc={rc} took {(time.time()-phase1_start)/60:.1f} min; remaining {(global_end_time-time.time())/60:.1f} min")


In [ ]:
# ---------------------------------------------------------------------------
# Phase 2 — Adaptive deep pass on starved/uncertain tasks if time remains (>20 min)
# ---------------------------------------------------------------------------
import os, sys, json, time, subprocess
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

def remaining():
    return global_end_time - time.time()

def resolve_challenges_path(rerun=False):
    fname = 'arc-agi_test_challenges.json' if rerun else 'arc-agi_evaluation_challenges.json'
    alt_fname = 'arc_agi_test_challenges.json' if rerun else 'arc_agi_evaluation_challenges.json'
    candidates = [
        f'/kaggle/input/competitions/arc-prize-2026-arc-agi-2/{fname}',
        f'/kaggle/input/arc-prize-2026-arc-agi-2/{fname}',
        f'/kaggle/input/arc-prize-2026/{fname}',
        f'/kaggle/input/{fname}',
        f'/kaggle/input/competitions/arc-prize-2026-arc-agi-2/{alt_fname}',
        f'/kaggle/input/arc-prize-2026-arc-agi-2/{alt_fname}',
        f'/kaggle/input/arc-prize-2026/{alt_fname}',
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    import glob
    for pattern in [f'/kaggle/input/**/{fname}', f'/kaggle/input/**/{alt_fname}']:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]
    return f'/kaggle/input/competitions/arc-prize-2026-arc-agi-2/{fname}'

test_file = resolve_challenges_path(rerun=True)
test_path = test_file if (RERUN or os.path.exists(test_file)) else resolve_challenges_path(rerun=False)

data = ArcDataset.from_file(test_path)
keys_in_scope = data.keys if (RERUN or os.path.exists(test_file)) else [k for k in data.keys if k in os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")]

dec = ArcDecoder(data.split_multi_replies(), n_guesses=2)
dec.load_decoded_results("/kaggle/inference_outputs")
dec.load_decoded_results("/kaggle/working/symbolic_outputs", run_name=".sym")
stats = dec.candidate_stats()

unprocessed, starved = [], {}
for k in keys_in_scope:
    n_out = len(data.queries[k]["test"])
    outs = [f"{k}_{i}" for i in range(n_out)]
    if not any(o in stats for o in outs):
        unprocessed.append(k)
        continue
    # Skip tasks that already have an exact verified symbolic rule solution
    has_sym_exact = all(any(s.get("is_symbolic_exact") for s in dec.decoded_results.get(o, {}).values()) for o in outs)
    if has_sym_exact:
        continue
    n_starved = sum(1 for o in outs if stats.get(o, {"unique": 0})["unique"] < 2)
    if n_starved:
        starved[k] = n_starved

print(f"[phase-2] unprocessed={len(unprocessed)} starved_tasks={len(starved)} remaining={remaining()/60:.1f} min")
json.dump({"unprocessed": unprocessed, "starved": starved}, open("/kaggle/working/phase2_plan.json", "w"))

base_env = dict(os.environ, UNSLOTH_DISABLE_STATISTICS="1", TRITON_PTXAS_PATH="/usr/local/cuda/bin/ptxas",
                OMP_NUM_THREADS="12", PYTHONHASHSEED="0", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", PYTORCH_ALLOC_CONF="expandable_segments:True", ARC_LORA_R="64", ARC_GRAD_CKPT="1")

def run_phase(name, keys, env_overrides, out_dir):
    if not keys or remaining() < 20 * 60:
        print(f"[phase-2:{name}] skipped (keys={len(keys)}, remaining={remaining()/60:.1f} min)")
        return
    keys_file = f"/kaggle/working/keys_{name}.json"
    json.dump(keys, open(keys_file, "w"))
    env = dict(base_env, ARC_OUT_DIR=out_dir, **{k: str(v) for k, v in env_overrides.items()})
    t = time.time()
    rc = subprocess.call([sys.executable, "starter.py", "--end-time", f"{global_end_time}", "--keys-file", keys_file, "--order", "file", "--skip-symbolic"], env=env)
    print(f"[phase-2:{name}] rc={rc} keys={len(keys)} took {(time.time()-t)/60:.1f} min; remaining {remaining()/60:.1f} min")

# (a) Catch-up with primary configuration
from starter import estimated_work
unprocessed = sorted(unprocessed, key=lambda k: estimated_work(data.queries[k]))
run_phase("catchup", unprocessed, {}, "/kaggle/inference_outputs")

# (b) Deep pass on starved outputs: most starved outputs first, then cheapest
deep_keys = sorted(starved, key=lambda k: (-starved[k], estimated_work(data.queries[k])))
if deep_keys and remaining() >= 20 * 60:
    per_task = max(600.0, min(2400.0, (remaining() - 300) * 4.0 / len(deep_keys)))
    deep_cfg = dict(ARC_LORA_SEED=137, ARC_TRAIN_AUG_SEED=17, ARC_EVAL_AUG_SEED=29, ARC_N_EVAL_AUG=3,
                    ARC_MIN_PROB=0.1, ARC_DFS_WINDOW=600, ARC_TASK_CAP=int(per_task), ARC_SCORE_SEED_OFFSET=7)
    print(f"[phase-2:deep] per-task cap {per_task:.0f}s, config {deep_cfg}")
    run_phase("deep", deep_keys, deep_cfg, "/kaggle/inference_outputs_deep")


In [ ]:
# ---------------------------------------------------------------------------
# Phase 3 — Final Selection, Diverse Attempt Generation, Schema Validation
# ---------------------------------------------------------------------------
import os, json, hashlib
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_v2

def resolve_file(filenames):
    candidates = []
    for fname in filenames:
        candidates.extend([
            f'/kaggle/input/competitions/arc-prize-2026-arc-agi-2/{fname}',
            f'/kaggle/input/arc-prize-2026-arc-agi-2/{fname}',
            f'/kaggle/input/arc-prize-2026/{fname}',
            f'/kaggle/input/{fname}',
        ])
    for c in candidates:
        if os.path.exists(c):
            return c
    import glob
    for fname in filenames:
        matches = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
        if matches:
            return matches[0]
    return candidates[0]

test_file = resolve_file(['arc-agi_test_challenges.json', 'arc_agi_test_challenges.json'])
test_path = test_file if (RERUN or os.path.exists(test_file)) else resolve_file(['arc-agi_evaluation_challenges.json', 'arc_agi_evaluation_challenges.json'])

data = ArcDataset.from_file(test_path)
sol_file = resolve_file(['arc-agi_evaluation_solutions.json', 'arc_agi_evaluation_solutions.json'])
if not RERUN and os.path.exists(sol_file):
    data = data.load_replies(sol_file)

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results("/kaggle/working/symbolic_outputs", run_name=".sym")
decoder.load_decoded_results("/kaggle/inference_outputs")
decoder.load_decoded_results("/kaggle/inference_outputs_deep", run_name=".deep")

# Generate orthogonal/diverse attempts for each test query
diverse_attempts = decoder.get_diverse_attempts(selection_algorithm=score_v2)

submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(data.queries[k]['test']))] for k in data.keys}

n_fallback = 0
for k in data.keys:
    for i, t in enumerate(data.queries[k]["test"]):
        subkey = f"{k}_{i}"
        target_entry = submission[k][i]
        if subkey in diverse_attempts:
            att1, att2 = diverse_attempts[subkey]
            target_entry["attempt_1"] = att1.tolist()
            target_entry["attempt_2"] = att2.tolist()
        else:
            target_entry["attempt_1"] = [[int(x) for x in row] for row in t["input"]]
            target_entry["attempt_2"] = [[0]]
            n_fallback += 1

        # Schema hardening
        for a in ("attempt_1", "attempt_2"):
            g = target_entry.get(a)
            ok = isinstance(g, list) and len(g) > 0 and len(g) <= 30 and all(isinstance(r, list) and len(r) == len(g[0]) and 0 < len(r) <= 30 for r in g)
            if not ok:
                if a == "attempt_1":
                    target_entry[a] = [[int(x) for x in row] for row in t["input"]]
                else:
                    target_entry[a] = [[0]]
            target_entry[a] = [[int(x) for x in row] for row in target_entry[a]]

        if target_entry["attempt_2"] == target_entry["attempt_1"]:
            target_entry["attempt_2"] = [[0]]

with open("/kaggle/working/submission.json", "w") as f:
    json.dump(submission, f)

print(f"*** wrote submission.json: {len(submission)} tasks, {sum(len(v) for v in submission.values())} outputs, {n_fallback} fallbacks, "
      f"sha256={hashlib.sha256(open('/kaggle/working/submission.json','rb').read()).hexdigest()[:12]}")

# Integrity verification
chk = json.load(open("/kaggle/working/submission.json"))
assert set(chk) == set(data.keys), "missing task ids"
for k in data.keys:
    assert len(chk[k]) == len(data.queries[k]["test"]), f"wrong #outputs for {k}"
    for e in chk[k]:
        assert "attempt_1" in e and "attempt_2" in e
        assert isinstance(e["attempt_1"], list) and isinstance(e["attempt_2"], list)
print("*** submission schema verified OK!")

if not RERUN and hasattr(data, "replies") and data.replies:
    val_score = data.validate_submission(chk)
    print(f"*** Validation score: {val_score:.2f} of {len(data.keys)} tasks")
